# 05 — Results Analysis & Cross-Task Demonstration

**Project:** CNN-Based Aerial Litter Detection for Sustainable Trail and Environmental Cleanup  
**Module:** ST7088CEM Artificial Neural Networks

This notebook pulls the whole project together: the consolidated cross-task
results and a qualitative demonstration on **unseen** (held-out test) aerial
images that shows the two tasks working as a pipeline — the tile classifier
flags litter-bearing regions (heatmap) and the detector localises individual
items (boxes).

## 1. Setup

Consolidation only needs the committed metric CSVs. Regenerating the
qualitative demo additionally needs both trained checkpoints
(`checkpoints/tile_cnn_best.pt`, `checkpoints/yolo_optimized.pt`) and the image
dataset — run it wherever those are available.

In [ ]:
# On Kaggle (Internet enabled), uncomment:
# !git clone -b feature/evaluation https://github.com/Sajan491/STW7088CEM-ANN-Assignment.git
# %cd STW7088CEM-ANN-Assignment
# %pip install -q ultralytics

import os, sys, platform
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))
print('Working directory:', Path.cwd())
print('Machine:', platform.node(), '|', platform.platform())

## 2. Consolidated cross-task results

In [ ]:
!python -m src.evaluation.consolidate

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

display(pd.read_csv('results/tables/final_summary.csv'))
display(IPImage('results/figures/final_summary.png', width=980))

## 3. Task 2 ablation (recap)

In [ ]:
display(pd.read_csv('results/tables/yolo_ablation.csv'))
display(IPImage('results/figures/yolo_ablation.png', width=900))

## 4. Qualitative demonstration on unseen images

Three panels per image: ground truth, YOLO detections, and the tile-classifier
litter heatmap. Regenerates if both checkpoints are present; otherwise displays
the committed demo figures.

In [ ]:
have_ckpts = (Path('checkpoints/tile_cnn_best.pt').exists()
              and Path('checkpoints/yolo_optimized.pt').exists())
if have_ckpts:
    !python -m src.evaluation.qualitative_demo --num-images 4
else:
    print('checkpoints not found — showing committed demo figures')

for p in sorted(Path('results/figures/qualitative').glob('demo_*.png')):
    display(IPImage(str(p), width=1100))

## 5. Conclusions for the report

- **Task 1 (custom CNN, from scratch):** the tile classifier reaches 90.5%
  accuracy and 0.75 F1 on held-out tiles (ROC-AUC 0.95), with recall (0.84)
  above precision (0.67) by design — the weighted loss favours flagging litter
  regions, appropriate for a screening stage.
- **Task 2 (transfer-learned YOLO):** the baseline reaches mAP@0.5 0.79; higher
  resolution and augmentation lift it to 0.86 (mAP@0.5:0.95 0.45 → 0.50).
  SAHI sliced inference did **not** help on this dataset — a documented negative
  result driven by false positives at 512 px slice scale.
- **Two-stage pipeline:** the qualitative demo shows the complementary roles —
  the lightweight tile classifier produces a coarse region-level litter map,
  and the detector localises individual items, an efficiency pattern relevant
  to on-drone deployment.
- **Reproducibility:** image-level splits, global seeds, config-driven runs, and
  unit tests (tile labelling, split leakage, model, conversion, SAHI helpers,
  heatmap) support end-to-end reproduction from the dataset link.